In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import subprocess
from eval_util_and_plots import *

In [ ]:
# run the rust code that generates the execution time data if the necessary files do not exist.
rust_project_dir = "../inc_search_eval"

data_dir = "inc_search_eval/paper_data" # This will generate plots using the raw data that generated the paper figures.
# data_dir = "inc_search_eval" # Uncomment this line to generate the data yourself. Once generated, the notebook wont overwrite it, you need to manually move/delete it to regenerate.

data_dir = data_dir.rstrip("/") + "/" # ensure it ends with a '/'


# These are the data sources needed for the plot
filenames = [
    data_dir+"results_execution_time_NCL5_maxkp256.csv.br",
    data_dir+"results_execution_time_NCL5_maxkp2048.csv.br",
    data_dir+"results_execution_time_NCL5_maxkp65535.csv.br",
    data_dir+"results_execution_time_NCL0_maxkp2048.csv.br",
    data_dir+"results_execution_time_NCL9_maxkp2048.csv.br",
]

# Check if ANY of the required files are missing
files_missing = any(not os.path.exists(f) for f in filenames)

if not files_missing:
    print("All result files already exist. Skipping Rust evaluation.")
else:
    # Ensure the target directory actually exists before running
    if not os.path.isdir(rust_project_dir):
        print(f"Error: Directory '{rust_project_dir}' not found.")
        sys.exit(1)

    print(f"Missing result files detected. Starting 'cargo run --release' in {rust_project_dir}...\n")

    try:
        # subprocess.run will wait for the command to finish.
        # It pipes standard output and error directly to your Python console.
        subprocess.run(
            ["cargo", "run", "--release"],
            cwd=rust_project_dir, # Sets the working directory for the command
            check=True            # Raises a CalledProcessError if the command fails
        )
        print("\nRust evaluation completed successfully.")
        
    except subprocess.CalledProcessError as e:
        print(f"\nExecution failed! Cargo exited with code {e.returncode}")
        sys.exit(e.returncode)
    except FileNotFoundError:
        print("\nError: 'cargo' command not found. Is Rust installed and in your system PATH?")
        sys.exit(1)

In [ ]:


filenames = [
    data_dir+"results_execution_time_NCL5_maxkp256.csv.br",
    data_dir+"results_execution_time_NCL5_maxkp2048.csv.br",
    data_dir+"results_execution_time_NCL5_maxkp65535.csv.br",
]

plt.figure(figsize=(8, 3))

for file in filenames:
    if not os.path.exists(file):
        print(f"Warning: {file} not found. Skipping.")
        continue

    try:
        df = read_brotli_csv(file)
        
        if 'execution_time' not in df.columns and 'execution_time_ns' not in df.columns:
            print(f"Column 'execution_time' not found in {file}. Skipping.")
            continue

        col_name = 'execution_time_ns' if 'execution_time_ns' in df.columns else 'execution_time'
        data = df[col_name].apply(parse_time_to_us).dropna()
        
        if len(data) == 0:
            print(f"No valid data in {file}. Skipping.")
            continue

        sorted_data = np.sort(data)
        
        # Calc CDF
        y = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
        
        label = file.split('_')[-1].replace('.csv.br', '')
        
        p = plt.plot(sorted_data, y, label=label, marker='.', linestyle='none', markersize=2, alpha=0.6)
        color = p[0].get_color()

        # 99.9 line
        p99_9 = np.percentile(sorted_data, 99.9)
        print(f"[{label}] 99.9th Percentile: {p99_9:.2f} µs")
        plt.axvline(x=p99_9, color=color, linestyle='--', linewidth=2.0, alpha=0.8)
        
    except Exception as e:
        print(f"Error processing {file}: {e}")

plt.plot([], [], color='black', linestyle='--', linewidth=2.0, label='99.9%')

plt.xscale('log')
plt.xlabel('Average Execution Time (µs, log)')
plt.ylabel('Cumulative Probability')
plt.title('CDF of Average Execution Time with NC_LIMIT=5')
plt.grid(True, which="both", ls="-", alpha=0.5)

plt.legend(markerscale=10)
plt.tight_layout()
plt.savefig("Search_cdf_avg_inference_time_NCL5.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

filenames = [
    data_dir+"results_execution_time_NCL0_maxkp2048.csv.br",
    data_dir+"results_execution_time_NCL5_maxkp2048.csv.br",
    data_dir+"results_execution_time_NCL9_maxkp2048.csv.br",
]

plt.figure(figsize=(8, 3))

for file in filenames:
    if not os.path.exists(file):
        print(f"Warning: {file} not found. Skipping.")
        continue

    try:
        df = read_brotli_csv(file)
        
        if 'execution_time' not in df.columns and 'execution_time_ns' not in df.columns:
            print(f"Column 'execution_time' not found in {file}. Skipping.")
            continue

        col_name = 'execution_time_ns' if 'execution_time_ns' in df.columns else 'execution_time'
        data = df[col_name].apply(parse_time_to_us).dropna()
        
        if len(data) == 0:
            print(f"No valid data in {file}. Skipping.")
            continue

        # CDF
        sorted_data = np.sort(data)
        y = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
        
        raw_part = file.split('_')[-2]
        label = raw_part.replace('NCL', 'NC_LIMIT ')
        
        p = plt.plot(sorted_data, y, label=label, marker='.', linestyle='none', markersize=2, alpha=0.6)
        color = p[0].get_color()

        # 99.9% line
        p99_9 = np.percentile(sorted_data, 99.9)
        print(f"[{label}] 99.9th Percentile: {p99_9:.2f} µs")
        plt.axvline(x=p99_9, color=color, linestyle='--', linewidth=2.0, alpha=0.8)
        
    except Exception as e:
        print(f"Error processing {file}: {e}")

plt.plot([], [], color='black', linestyle='--', linewidth=2.0, label='99.9%')

plt.xscale('log')
plt.xlabel('Average Execution Time (µs, log)')
plt.ylabel('Cumulative Probability')
plt.title('CDF of Average Execution Time with max_kp=2048')
plt.grid(True, which="both", ls="-", alpha=0.5)

plt.legend(markerscale=10) 
plt.tight_layout()
plt.savefig("Search_cdf_avg_inference_time_maxkp2048.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
#decompress_brotli("inc_search_eval/results_execution_time_NCL0_maxkp2048.csv.br", "out.csv")